# Moosic — 04. DBSCAN

Density-based clustering: no k to specify, but points that don't belong to any dense region are labelled noise (`-1`) rather than forced into a cluster. Starts fresh from the CSV.

**Scaling note:** the course example uses `StandardScaler`. We're using `MinMaxScaler` instead, matching notebooks `01`-`03`, for a fair head-to-head with K-Means. Confirmed with the instructor this doesn't undermine the comparison, since silhouette score is a normalized ratio, not a raw distance value.

**Structure note (cleaned up after a real mix-up):** every side-experiment below writes to its own uniquely-named variable. Only **Step 5** writes to the shared `df['dbscan_cluster']` column that every later step (histogram, t-SNE, noise investigation, comparison) reads from. This is deliberate — an earlier version let a side-experiment (`4a`, testing `min_samples=7`) compute its own labels without saving them anywhere shared, so later steps kept silently reading a different, older run instead. Rebuilt so that can't happen again.


## 0. Success Criteria — Defined Before Comparing Anything

**"Good playlist" has no single universal definition — that's not a gap in this project, it's a real fact about the problem.** Rather than search for one metric that settles it, we define two explicit axes here, up front, and score every algorithm against both.

1. **Coverage** — % of the 5000 songs actually placed into some cluster.
2. **Coherence** — how tight and musically defensible the resulting clusters are (silhouette score on non-noise points, plus manual spot-checks).

K-Means (as built in `01`-`03`) optimizes for coverage. DBSCAN optimizes for coherence over coverage. Neither is more "correct" — the report's conclusion states which philosophy fits Moosic's actual priorities.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data & Scale

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

scaler = MinMaxScaler().set_output(transform="pandas")
scaled = scaler.fit_transform(df[features])

print(f"Shape: {df.shape}")

## 3. Finding Epsilon — k-distance Graph

In [ ]:
N_NEIGHBORS = 3

neighbours = NearestNeighbors(n_neighbors=N_NEIGHBORS)
neighbours.fit(scaled)
distances, indices = neighbours.kneighbors(scaled)
kth_distances = distances[:, N_NEIGHBORS - 1]
sorted_distances = sorted(kth_distances)

(
    sns.relplot(kind="line", x=range(len(sorted_distances)), y=sorted_distances, aspect=1.6)
    .set_axis_labels("Data points sorted by distance", f"{N_NEIGHBORS}-th Nearest Neighbour Distance")
    .set(title=f"k-distance Graph (n_neighbors={N_NEIGHBORS})")
)
plt.savefig("../outputs/04_kdistance_n3.png", dpi=150, bbox_inches="tight")
plt.show()

### 3a. Experiment — `n_neighbors = 7`, matching our feature count (per instructor guidance, simpler than a 2x-dimensionality heuristic)


In [ ]:
N_NEIGHBORS_3A = 7

neighbours_3a = NearestNeighbors(n_neighbors=N_NEIGHBORS_3A)
neighbours_3a.fit(scaled)
distances_3a, _ = neighbours_3a.kneighbors(scaled)
kth_distances_3a = distances_3a[:, N_NEIGHBORS_3A - 1]
sorted_distances_3a = sorted(kth_distances_3a)

(
    sns.relplot(kind="line", x=range(len(sorted_distances_3a)), y=sorted_distances_3a, aspect=1.6)
    .set_axis_labels("Data points sorted by distance", f"{N_NEIGHBORS_3A}-th Nearest Neighbour Distance")
    .set(title=f"k-distance Graph (n_neighbors={N_NEIGHBORS_3A})")
)
plt.savefig("../outputs/04_kdistance_n7.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Parameter Sweep

Sweeps `eps` across a wide range at a fixed `min_samples`, to map the full coverage/coherence tradeoff rather than commit to one guessed value. Results from an earlier full run are recorded in the markdown cell below the code, for reference without needing to re-run everything.


In [ ]:
def eps_sweep(eps_values, min_samples):
    results = []
    for eps in eps_values:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(scaled)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        coverage_pct = 100 * (1 - n_noise / len(labels))

        non_noise = labels != -1
        if n_clusters > 1 and non_noise.sum() > 1:
            sil = silhouette_score(scaled[non_noise], labels[non_noise])
        else:
            sil = None

        results.append({
            "eps": eps, "clusters": n_clusters, "noise": n_noise,
            "coverage_%": round(coverage_pct, 1),
            "silhouette": round(sil, 3) if sil is not None else None
        })
    return pd.DataFrame(results)

sweep_min_samples_7 = eps_sweep([0.03, 0.05, 0.07, 0.09, 0.11, 0.13, 0.15, 0.17, 0.20, 0.22, 0.25, 0.30, 0.40, 0.50], min_samples=7)
print(sweep_min_samples_7)

**Recorded result (MinMaxScaler, min_samples=7):** no `eps` in this range gives many reasonably-sized clusters, good coherence, and high coverage simultaneously. Mid-range eps (0.09-0.20) has acceptable coverage but negative silhouette (fragmented, overlapping regions). Coverage and silhouette both look good only at eps=0.20-0.22 (92-94%, silhouette ~0.45) — but this is misleading (see Step 5). Past ~0.25, total collapse to one cluster.

**Confirmed independently with `StandardScaler`** (different absolute eps range, 0.1-2.0, same shape of result) and with **very low `min_samples` (1-3)**, which causes a different failure mode — chaining — where sparse bridging points let clusters merge into 2-3 giant blobs with negative silhouette, same underlying "no viable middle ground" conclusion from the opposite direction.


## 5. Chosen Configuration — the Canonical Run

`eps=0.20, min_samples=7` — the most thoroughly investigated configuration, and the one every step from here on depends on via `df['dbscan_cluster']`.


In [ ]:
EPS = 0.20
MIN_SAMPLES = 7

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
df['dbscan_cluster'] = dbscan.fit_predict(scaled)

n_clusters = len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'].values else 0)
n_noise = (df['dbscan_cluster'] == -1).sum()

print(f"eps={EPS}, min_samples={MIN_SAMPLES}")
print(f"Clusters found: {n_clusters}")
print(f"Noise points: {n_noise} ({100*n_noise/len(df):.1f}% of dataset)")
print(df['dbscan_cluster'].value_counts().sort_index())

### 5a. Side-check — `min_samples=3`, same eps (own variables, does NOT touch `df['dbscan_cluster']`)

Tests whether a smaller `min_samples` gives a genuinely better result, or just decorates the same failure mode with more clusters.


In [ ]:
dbscan_ms3 = DBSCAN(eps=0.20, min_samples=3)
labels_ms3 = dbscan_ms3.fit_predict(scaled)

print(pd.Series(labels_ms3).value_counts().sort_index())

**Result, confirmed:** `min_samples=3` gives 11 labels instead of 2, but the size breakdown is `{0: 4919, 1-10: 3-9 each, noise: 274}` — 94% of the dataset still lands in one mega-cluster; the extra 10 "clusters" are near-empty scraps. **Same failure mode as the canonical run, not a real improvement.** Not adopted.


## 6. Cluster Size Distribution

In [ ]:
cluster_sizes = df[df['dbscan_cluster'] != -1]['dbscan_cluster'].value_counts().sort_index()
print(cluster_sizes)

plt.figure(figsize=(10, 5))
plt.hist(cluster_sizes, bins=30, color='steelblue', edgecolor='black')
plt.xlabel('Cluster size'); plt.ylabel('Number of clusters')
plt.title(f'DBSCAN cluster size distribution ({len(cluster_sizes)} clusters, {n_noise} noise points excluded)')
plt.tight_layout()
plt.savefig("../outputs/04_dbscan_size_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Visualise with t-SNE

In [ ]:
tsne = TSNE(n_components=2, random_state=RANDOM_STATE)
tsne_results = tsne.fit_transform(scaled)
tsne_results['Cluster'] = df['dbscan_cluster'].astype("string")
tsne_results.loc[tsne_results['Cluster'] == '-1', 'Cluster'] = 'Outlier'

sns.relplot(
    data=tsne_results,
    x=tsne_results.columns[0], y=tsne_results.columns[1],
    hue='Cluster', palette='colorblind', s=25
).set(title="DBSCAN Clustering Visualisation with t-SNE")
plt.savefig("../outputs/04_dbscan_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

**Confirmed visually:** one dominant blue mass, a genuinely separate lobe (the classical/instrumental cluster — pulled visually apart by high acousticness/instrumentalness), noise points sitting mostly along the edge of the main mass rather than forming their own island — consistent with them being boundary/blend cases.


## 8. Investigate the Noise Points

In [ ]:
noise = df[df['dbscan_cluster'] == -1]
mainstream_avg = df[df['dbscan_cluster'] == 0][features].mean()
classical_avg = df[df['dbscan_cluster'] == 1][features].mean()

print(pd.DataFrame({
    "mainstream_avg": mainstream_avg,
    "classical_avg": classical_avg,
    "noise_avg": noise[features].mean()
}).round(3))

**Read on 5 of 7 features:** noise sits between mainstream and classical — consistent with genuine boundary/blend cases DBSCAN correctly refused to force into either group. **Tempo and speechiness are exceptions** — noise is higher than *both* clusters on these two, not between them. That's a different shape entirely: not a blend, but a distinct third pocket.


In [ ]:
print(f"Noise points with speechiness > 0.33 (Spotify's rap/spoken-word threshold): "
      f"{(noise['speechiness'] > 0.33).sum()} out of {len(noise)}")

print("\nHighest-speechiness noise songs:")
print(noise.nlargest(15, 'speechiness')[['name', 'speechiness', 'tempo']].to_string(index=False))

**Confirmed by title, not just numbers:** "This Is Why I'm Hot," "Candy Shop," "Bad Boy for Life," "I Need A Girl Pt. II" — genuine, recognizable hip-hop/rap tracks. A real pocket, not a statistical artifact.


In [ ]:
noise_scaled = scaled.loc[noise.index]

print("=== Sub-clustering the canonical (417-song) noise set ===")
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(noise_scaled)
    sil = silhouette_score(noise_scaled, labels)
    print(f"k={k}: silhouette={sil:.3f}, sizes={pd.Series(labels).value_counts().sort_index().tolist()}")

In [ ]:
BEST_NOISE_K = 3  # best silhouette from the sweep above

noise = noise.copy()
noise['sub_cluster'] = KMeans(n_clusters=BEST_NOISE_K, random_state=RANDOM_STATE, n_init=10).fit_predict(noise_scaled)

for sub in range(BEST_NOISE_K):
    group = noise[noise['sub_cluster'] == sub]
    print(f"\n--- Sub-cluster {sub} ({len(group)} songs) ---")
    print(group[features].mean().round(3))

**Check:** the sub-cluster with the highest `speechiness` and `tempo`, clearly above the noise-wide averages printed further up, is the confirmed rap/hip-hop pocket — hidden inside DBSCAN's noise bucket, too small and too differently-shaped to earn its own dense region, but real and coherent all the same.


### 8a. Bonus — independent replication via the `min_samples=3` noise set (274 songs)

A separate sub-clustering run, done on Step 5a's *different* noise set, found essentially the same signal independently: one sub-cluster (115 songs) with clearly elevated speechiness (0.196) and tempo (136.6) relative to its own group's average — the rap-pocket finding replicated via a different DBSCAN configuration, not a fluke of one parameter choice. The other two sub-clusters split further into a classical/acoustic-leaning group (high acousticness/instrumentalness, low energy) and a separate energetic-but-instrumental group (high energy and instrumentalness together, low acousticness — plausibly electronic/dance instrumentals) — a third distinct pocket worth a mention, not fully investigated further here.


## 9. Data Quality Aside — Resolved Investigation

Separate from DBSCAN tuning: while inspecting the classical/instrumental cluster, "Moves Like Jagger" turned up as a misfit sample. Investigated and resolved — kept here for the record, not part of the DBSCAN parameter story above.

**Finding:** the dataset contains a small number of genuine, unlabelled instrumental/piano cover versions of well-known pop songs, sharing exact titles with the originals (confirmed for both "Moves Like Jagger" and "Shape of You" — each has two real, distinct rows). Not a data-quality bug. A positive result: the audio features correctly separated the pop original from its cover in both cases, despite the identical title giving no hint.

**Methodology note kept for the record:** the first duplicate-title check averaged **unscaled** feature standard deviations, which let `tempo`'s much larger numeric range (60-200 BPM vs. 0-1 for other features) silently dominate the "suspicious" ranking. Corrected by running the check on scaled data instead.


In [ ]:
# Corrected version — scaled, not raw
scaled_with_names = scaled.copy()
scaled_with_names['name'] = df['name'].values

dupes_scaled = scaled_with_names[scaled_with_names.duplicated('name', keep=False)]
suspicious = []
for name, group in dupes_scaled.groupby('name'):
    if len(group) > 1:
        spread = group[features].std().mean()
        if spread > 0.25:
            suspicious.append((name, len(group), round(spread, 3)))

pd.DataFrame(suspicious, columns=['name', 'count', 'avg_feature_std']).sort_values('avg_feature_std', ascending=False).head(20)

## 10. Compare Against the K-Means Baseline

In [ ]:
non_noise_mask = df['dbscan_cluster'] != -1
if df['dbscan_cluster'].nunique() > 1 and non_noise_mask.sum() > 1:
    dbscan_silhouette = silhouette_score(scaled[non_noise_mask], df.loc[non_noise_mask, 'dbscan_cluster'])
else:
    dbscan_silhouette = None

kmeans_baseline = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_baseline.fit_predict(scaled)
kmeans_silhouette = silhouette_score(scaled, kmeans_labels)

comparison = pd.DataFrame({
    "Method": ["K-Means (k=8)", "DBSCAN"],
    "Clusters": [8, n_clusters],
    "Noise/unclustered": [0, n_noise],
    "Silhouette (non-noise only)": [round(kmeans_silhouette, 3), round(dbscan_silhouette, 3) if dbscan_silhouette else "N/A"]
})
comparison

**Honest framing:** these silhouette scores aren't apples-to-apples — DBSCAN's excludes noise, K-Means's doesn't. Report both the score **and** the noise percentage together, never the score alone.


## 11. Verdict

No tested `eps`/`min_samples` combination, under either scaler, gives many reasonably-sized clusters, good coherence, and high coverage at the same time — confirmed across a full sweep, a second scaler, and both very low and moderate `min_samples`. This is a real, reproducible property of the dataset's feature space (a continuous mainstream blend with no natural density gaps), not a tuning failure.

DBSCAN's genuine value here was diagnostic, not as a playlist generator: it cleanly isolated one real distinct pocket (classical/instrumental) and, on closer inspection of its noise bucket, revealed a second real pocket (rap/hip-hop) too small or oddly-shaped to form its own dense region under any tested setting.

**Standing recommendation unchanged:** the K-Means recursive split/merge pipeline (notebooks `01`-`03`) remains the practical choice for Moosic, pending the Agglomerative Clustering comparison.

---
**Next:** Agglomerative Clustering, compared against both K-Means and DBSCAN on the same lens.
